# Explore: ingest · retrieve · generate
Walks through every public function in `ingest.py`, `retrieve.py`, and `generator.py`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import ingest, retrieve, generator

## 1. Loaded documents (one per file in `data/`)

In [ ]:
documents = ingest.load_documents()
print(f"{len(documents)} documents loaded\n")

for doc in documents:
    print(f"source={doc.metadata['source']!r} type={doc.metadata['type']!r} "
          f"skills={doc.metadata['skills']!r} chars={len(doc.page_content)}")

## 1b. Loaded PDF cover letters (`data/cover letters/*.pdf`)
Metadata (`type`, `company`, `skills`) is now extracted per PDF by calling `gpt-4o-mini`
(see `ingest.extract_metadata_llm`). This makes one API call per PDF, so re-running this
cell costs a small amount and requires `OPENAI_API_KEY` in `.env`. The outputs below are
from a previous run and will refresh (including `company`/`skills` values) the next time
this cell executes.

In [ ]:
pdf_documents = ingest.load_pdf_documents()
print(f"{len(pdf_documents)} PDF documents loaded\n")

for doc in pdf_documents:
    print(f"source={doc.metadata['source']!r} type={doc.metadata['type']!r} "
          f"company={doc.metadata['company']!r} skills={doc.metadata['skills']!r} "
          f"chars={len(doc.page_content)}")

## 2. Chunked documents (token-based, 500/50 overlap)

In [ ]:
chunks = ingest.chunk_documents(documents + pdf_documents)
print(f"{len(chunks)} chunks produced\n")

from collections import Counter
counts = Counter(c.metadata["source"] for c in chunks)
for source, count in counts.items():
    print(f"{source}: {count} chunk(s)")

In [ ]:
# Peek at one chunk's content
sample = chunks[0]
print(sample.metadata)
print("---")
print(sample.page_content[:500])

## 3. (Optional) Run full ingest — embeds chunks via OpenAI and persists to `chroma_db/`
`ingest.ingest()` loads every PDF in `cover letters/`, `projects/`, and `cv/`, calling
`gpt-4o-mini` once per PDF to extract `type`/`company`/`skills` metadata, then embeds the
chunks with OpenAI. This costs more than before (one chat completion per PDF, in addition
to the embeddings) and requires `OPENAI_API_KEY` in `.env`.

In [ ]:
vectorstore = ingest.ingest()

## 4. Retrieval — `retrieve.get_retriever()` + `retrieve.retrieve()`
Requires `chroma_db/` to exist (run cell 3 first). Returns plain strings, one per chunk.

In [ ]:
job_description2 = """
We are looking for a Senior Data Scientist with experience in
recommendation systems, production ML pipelines, and NLP.
Experience with cloud platforms (AWS/Azure) and MLOps tooling preferred.
"""
job_description = """
We are looking for a Senior Data Scientist with experience in
organising meetups and MLOps tooling.
"""

chunks = retrieve.retrieve(job_description, k=4)
print(f"{len(chunks)} chunks retrieved\n")
for i, chunk in enumerate(chunks, 1):
    print(f"--- chunk {i} ---")
    print(chunk)
    print()

## 5. Generation — `generator.generate()`
Joins retrieved chunks into context and asks `gpt-4o-mini` for one tailored cover letter paragraph.
Requires `OPENAI_API_KEY` in `.env`.

In [ ]:
context = "\n\n".join(chunks)  # chunks retrieved in section 4

paragraph = generator.generate(context=context, job_description=job_description)
print(paragraph)

In [ ]:
test_queries = [
    "team lead and stakeholder communication"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    chunks = retrieve.retrieve(query, k=2)
    for c in chunks:
        print(f"  -> {c}")